In [9]:
#🔧 Célula 0: Setup geral
# Instala os recursos e carrega tudo o que as 3 aulas usam.

import re, unicodedata
import numpy as np
import pandas as pd
from collections import Counter

import nltk
for r in ['punkt', 'punkt_tab', 'stopwords', 'rslp']:
    nltk.download(r, quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

import spacy
try:
    nlp = spacy.load('pt_core_news_lg')
except OSError:
    from spacy.cli import download
    download('pt_core_news_lg')
    nlp = spacy.load('pt_core_news_lg')

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

STOP_WORDS_PT = set(stopwords.words('portuguese'))
STOP_WORDS_PT = set(stopwords.words('portuguese'))
STOP_WORDS_SENT = STOP_WORDS_PT - {'não', 'nem', 'muito', 'mais', 'menos'}  # 👈 nova
stemmer = RSLPStemmer()
print('Ambiente pronto ✅')

Ambiente pronto ✅


In [10]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


#📘 Aula 1: Fundamentos de NLP

In [11]:
#Célula 1: Polissemia (Bloco 01)
#Mostra que no BoW a palavra "banco" é idêntica nos 3 sentidos, porque o contexto se perde.

corpus = [
    "O banco aprovou meu empréstimo ontem",
    "Sentei no banco da praça para descansar",
    "O banco de dados PostgreSQL caiu novamente",
]
vec = CountVectorizer()
X = vec.fit_transform(corpus)
print('Vocabulário:', vec.get_feature_names_out())
print(X.toarray())


Vocabulário: ['aprovou' 'banco' 'caiu' 'da' 'dados' 'de' 'descansar' 'empréstimo' 'meu'
 'no' 'novamente' 'ontem' 'para' 'postgresql' 'praça' 'sentei']
[[1 1 0 0 0 0 0 1 1 0 0 1 0 0 0 0]
 [0 1 0 1 0 0 1 0 0 1 0 0 1 0 1 1]
 [0 1 1 0 1 1 0 0 0 0 1 0 0 1 0 0]]


In [12]:
#Célula 2: Corpus de reviews (Bloco 03)
#Cria o corpus que as próximas células da Aula 1 usam. O último review é uma ironia.

corpus = [
    "O produto é excelente, recomendo muito! Chegou rápido.",
    "Péssimo, o aparelho veio com defeito. Não comprem.",
    "Ótima qualidade pelo preço. Entrega rápida e bem embalado.",
    "Adorei! Funciona perfeitamente, superou minhas expectativas.",
    "Esperei 3 horas no telefone. Adorei o atendimento.",
]

In [13]:
#Célula 3: Tokenização e Lei de Zipf (Bloco 04)
#Conta os tokens. Você vai ver que stop words e pontuação dominam o topo.

todos_tokens = []
for doc in corpus:
    todos_tokens.extend(word_tokenize(doc.lower(), language='portuguese'))

print(f'Total de tokens: {len(todos_tokens)}')
print(f'Vocabulário (únicos): {len(set(todos_tokens))}')
for palavra, freq in Counter(todos_tokens).most_common(10):
    print(f'  {palavra:15s} → {freq}')


Total de tokens: 52
Vocabulário (únicos): 39
  .               → 8
  o               → 3
  ,               → 3
  !               → 2
  adorei          → 2
  produto         → 1
  é               → 1
  excelente       → 1
  recomendo       → 1
  muito           → 1


In [14]:
#Célula 4: Análise linguística com spaCy (Bloco 05)
#Para cada token, mostra o lema, a classe gramatical (POS) e a dependência sintática.

doc = nlp("O João comprou um livro na Amazon ontem.")
print(f"{'TOKEN':12s} {'LEMA':12s} {'POS':10s} {'DEP':10s}")
print('-' * 50)
for t in doc:
    print(f"{t.text:12s} {t.lemma_:12s} {t.pos_:10s} {t.dep_:10s}")


TOKEN        LEMA         POS        DEP       
--------------------------------------------------
O            o            DET        det       
João         João         PROPN      nsubj     
comprou      comprar      VERB       ROOT      
um           um           DET        det       
livro        livro        NOUN       obj       
na           em o         ADP        case      
Amazon       Amazon       PROPN      obl       
ontem        ontem        ADV        advmod    
.            .            PUNCT      punct     


In [15]:
#Célula 5: NER (Bloco 06)
#Encontra as entidades: pessoas (PER), locais (LOC) etc.

texto = ("Leonardo da Vinci nasceu em Vinci, Itália, em 1452. "
         "Foi pintor, cientista e trabalhou para Lorenzo de Médici. "
         "Morreu em Amboise, na França.")
for ent in nlp(texto).ents:
    print(f'  {ent.text:25s} → {ent.label_}')

  Leonardo da Vinci         → PER
  Vinci                     → LOC
  Itália                    → LOC
  Lorenzo de Médici         → PER
  Amboise                   → LOC
  França                    → LOC


In [16]:
#Célula 6: Primeira vetorização (Bloco 07)
#Gera o BoW sem stop words e mostra a matriz documento-termo.

vec = CountVectorizer(stop_words=list(STOP_WORDS_SENT))
X = vec.fit_transform(corpus)
print(f'Shape: {X.shape}')
df_dt = pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())
print(df_dt.iloc[:, :8])


Shape: (5, 28)
   adorei  aparelho  atendimento  bem  chegou  comprem  defeito  embalado
0       0         0            0    0       1        0        0         0
1       0         1            0    0       0        1        1         0
2       0         0            0    1       0        0        0         1
3       1         0            0    0       0        0        0         0
4       1         0            1    0       0        0        0         0


# 📗 Aula 2: Pré-processamento

In [17]:
#Célula 7: Regex básico (Bloco 01)
#Remove HTML, URLs, e-mails e símbolos, um passo de cada vez.

texto = "<p>Adorei!! https://link.com vc tem que ver isso :)</p>"
limpo = re.sub(r'<[^>]+>', ' ', texto)
limpo = re.sub(r'http\S+|www\.\S+', ' ', limpo)
limpo = re.sub(r'\S+@\S+', ' ', limpo)
limpo = re.sub(r'[^a-záàâãäéèêëíìîïóòôõöúùûüç\s]', ' ', limpo.lower())
limpo = re.sub(r'\s+', ' ', limpo).strip()
print(limpo)


adorei vc tem que ver isso


In [18]:
#Célula 8: Reviews "sujas" e contrações (Blocos 02 e 03)
#Expande o internetês ("vc" vira "você"). O CONTRACTIONS é obrigatório para a Célula 10.

reviews = [
    "<p>ADOREI o produto!! vc precisa cmprar tbm 😍</p>",
    "Pessimo!! veio quebrado >:( https://reclame.com",
    "Mt bom mesmo, entrega rapida! 5 estrelas pra esse vendedor.",
    "Não gostei :( voltei o item, atendimento foi pessimo tbm.",
]
CONTRACTIONS = {
    'vc': 'você', 'tb': 'também', 'tbm': 'também', 'pq': 'por que',
    'obg': 'obrigado', 'blz': 'beleza', 'td': 'tudo', 'msm': 'mesmo',
    'vlw': 'valeu', 'n': 'não', 'naum': 'não', 'cmprar': 'comprar',
    'mt': 'muito', 'pra': 'para'
}
def limpar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'<[^>]+>', ' ', texto)
    texto = re.sub(r'http\S+|www\.\S+', ' ', texto)
    texto = re.sub(r'\S+@\S+', ' ', texto)
    return ' '.join(CONTRACTIONS.get(p, p) for p in texto.split())

print(limpar_texto(reviews[0]))

adorei o produto!! você precisa comprar também 😍


In [19]:
#Célula 9: Remover acentos (Bloco 04)
#Ajuda em buscas, mas atrapalha o NER. Por isso é uma decisão que depende do uso.

def remove_acentos(texto):
    nfkd = unicodedata.normalize('NFKD', texto)
    return ''.join(c for c in nfkd if not unicodedata.combining(c))

for p in ['São Paulo', 'experiência', 'coração']:
    print(remove_acentos(p))

Sao Paulo
experiencia
coracao


In [20]:
#Célula 10: limpar_normalizar() (Bloco 05) ⭐
#É a função central. A célula do Olist depende dela.

def limpar_normalizar(texto, remove_acentos=False):
    if not isinstance(texto, str):
        return ''
    texto = texto.lower()
    texto = re.sub(r'<[^>]+>', ' ', texto)
    texto = re.sub(r'http\S+|www\.\S+', ' ', texto)
    texto = re.sub(r'\S+@\S+', ' ', texto)
    texto = ' '.join(CONTRACTIONS.get(p, p) for p in texto.split())
    texto = re.sub(r'[^a-záàâãäéèêëíìîïóòôõöúùûüç\s.,!?]', ' ', texto)
    if remove_acentos:
        nfkd = unicodedata.normalize('NFKD', texto)
        texto = ''.join(c for c in nfkd if not unicodedata.combining(c))
    return re.sub(r'\s+', ' ', texto).strip()

print(limpar_normalizar('<p>Adorei o produto!! https://exemplo.com VC PRECISA TBM!!!</p>'))

adorei o produto!! você precisa tbm!!!


In [21]:
#Célula 11: Três formas de tokenizar (Blocos 06, 07 e 08)
#Compara split(), NLTK e spaCy. O split() deixa a pontuação grudada nas palavras.

texto = "O Sr. Silva, hoje, tem R$ 1.250,99 no banco! Adorei."
print('split():', texto.split())
print('NLTK   :', word_tokenize(texto, language='portuguese'))
for tok in nlp(texto):
    print(f"{tok.text:<12} {tok.lemma_:<12} {tok.pos_}")

split(): ['O', 'Sr.', 'Silva,', 'hoje,', 'tem', 'R$', '1.250,99', 'no', 'banco!', 'Adorei.']
NLTK   : ['O', 'Sr.', 'Silva', ',', 'hoje', ',', 'tem', 'R', '$', '1.250,99', 'no', 'banco', '!', 'Adorei', '.']
O            o            DET
Sr.          Sr.          NOUN
Silva        Silva        PROPN
,            ,            PUNCT
hoje         hoje         ADV
,            ,            PUNCT
tem          ter          VERB
R$           R$           SYM
1.250,99     1.250,99     NUM
no           em o         ADP
banco        banco        NOUN
!            !            PUNCT
Adorei       Adorei       VERB
.            .            PUNCT


In [22]:
#Célula 12: Tokenização de frases (Bloco 09)
#Separa o texto em frases, com NLTK e com spaCy.

texto = "Adorei o produto. Chegou rápido! Recomendo a todos. Mas o preço foi alto..."
for i, f in enumerate(sent_tokenize(texto, language='portuguese'), 1):
    print(f'{i}. {f}')
for sent in nlp(texto).sents:
    print('spaCy:', sent.text)

1. Adorei o produto.
2. Chegou rápido!
3. Recomendo a todos.
4. Mas o preço foi alto...
spaCy: Adorei o produto.
spaCy: Chegou rápido!
spaCy: Recomendo a todos.
spaCy: Mas o preço foi alto...


In [23]:
#Célula 13: Subwords do BERTimbau (Bloco 11)
#Mostra como os Transformers quebram palavras raras em pedaços.

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')
print(tok.tokenize('correríamos pela praia'))
print(tok.tokenize('inconstitucionalissimamente'))

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

['correr', '##íamos', 'pela', 'praia']
['incons', '##titu', '##cional', '##iss', '##imamente']


In [24]:
#Célula 14: Stemming x Lematização (Blocos 12 e 13)
#O stemming corta a palavra ("corr"). A lematização devolve uma palavra válida ("correr").

for p in ['correndo', 'correu', 'maravilhoso', 'meninas', 'gatos']:
    print(f'{p:<15} → {stemmer.stem(p)}')

for t in nlp('As crianças estavam correndo felizes pela praia maravilhosa.'):
    print(f'{t.text:<14} → {t.lemma_}')


correndo        → corr
correu          → corr
maravilhoso     → maravilh
meninas         → menin
gatos           → gat
As             → o
crianças       → criança
estavam        → estar
correndo       → correr
felizes        → feliz
pela           → por o
praia          → praia
maravilhosa    → maravilhoso
.              → .


In [25]:
#Célula 15: Stop words e pipeline final (Blocos 14 e 15)
#Limpa, tokeniza, tira as stop words e aplica o stem, tudo numa função só.

import re

def limpar_normalizar(texto):
    texto = texto.lower()
    texto = re.sub(r'http\S+|www\.\S+', ' ', texto)   # URLs
    texto = re.sub(r'[^a-záàâãéêíóôõúç\s]', ' ', texto)  # pontuação/números
    return re.sub(r'\s+', ' ', texto).strip()

print(f'Total: {len(STOP_WORDS_PT)} stop words')

def pipeline_nlp(textos):
    resultado = []
    for texto in textos:
        tokens = word_tokenize(limpar_normalizar(texto), language='portuguese')
        tokens = [stemmer.stem(t) for t in tokens
                 if t not in STOP_WORDS_SENT and (len(t) > 2 or t in {'não', 'nem'})]
        resultado.append(tokens)
    return resultado

print(pipeline_nlp(['Adorei o produto, entrega rapidíssima!',
                    'Não gostei, veio quebrado e atrasado.']))


Total: 207 stop words
[['ador', 'produt', 'entreg', 'rapid'], ['não', 'gost', 'vei', 'quebr', 'atras']]


#📙 Aula 3: BoW e TF-IDF

In [26]:
#Célula 16: BoW manual (Bloco 01)
#Monta a matriz documento-termo "na mão", para entender o que o sklearn faz por dentro.

corpus = ['Adorei o produto, entrega rápida',
          'Produto chegou quebrado',
          'Entrega rápida, atendimento bom']
docs_tokens = [d.lower().replace(',', '').split() for d in corpus]
vocab = sorted(set(t for d in docs_tokens for t in d))
matriz = [[Counter(d).get(p, 0) for p in vocab] for d in docs_tokens]
print(pd.DataFrame(matriz, columns=vocab))

   adorei  atendimento  bom  chegou  entrega  o  produto  quebrado  rápida
0       1            0    0       0        1  1        1         0       1
1       0            0    0       1        0  0        1         1       0
2       0            1    1       0        1  0        0         0       1


In [27]:
#Célula 17: CountVectorizer e esparsidade (Blocos 02 e 04)
#Faz o mesmo BoW com o sklearn e mostra a porcentagem de zeros na matriz.

corpus = ['Adorei o produto, entrega rápida e qualidade boa.',
          'Produto chegou quebrado, qualidade péssima.',
          'Entrega rápida, atendimento bom, recomendo!',
          'Produto bom mas atendimento péssimo.']
vectorizer = CountVectorizer(max_features=20)
X = vectorizer.fit_transform(corpus)
print(pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out()))
print(f'Shape: {X.shape} | Não-zeros: {X.nnz}')
print(f'Esparsidade: {1 - X.nnz / (X.shape[0] * X.shape[1]):.2%}')

   adorei  atendimento  boa  bom  chegou  entrega  mas  produto  péssima  \
0       1            0    1    0       0        1    0        1        0   
1       0            0    0    0       1        0    0        1        1   
2       0            1    0    1       0        1    0        0        0   
3       0            1    0    1       0        0    1        1        0   

   péssimo  qualidade  quebrado  recomendo  rápida  
0        0          1         0          0       1  
1        0          1         1          0       0  
2        0          0         0          1       1  
3        1          0         0          0       0  
Shape: (4, 14) | Não-zeros: 21
Esparsidade: 62.50%


In [28]:
#Célula 17: CountVectorizer e esparsidade (Blocos 02 e 04)
#Faz o mesmo BoW com o sklearn e mostra a porcentagem de zeros na matriz.

corpus = ['Adorei o produto, entrega rápida e qualidade boa.',
          'Produto chegou quebrado, qualidade péssima.',
          'Entrega rápida, atendimento bom, recomendo!',
          'Produto bom mas atendimento péssimo.']
vectorizer = CountVectorizer(max_features=20)
X = vectorizer.fit_transform(corpus)
print(pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out()))
print(f'Shape: {X.shape} | Não-zeros: {X.nnz}')
print(f'Esparsidade: {1 - X.nnz / (X.shape[0] * X.shape[1]):.2%}')


   adorei  atendimento  boa  bom  chegou  entrega  mas  produto  péssima  \
0       1            0    1    0       0        1    0        1        0   
1       0            0    0    0       1        0    0        1        1   
2       0            1    0    1       0        1    0        0        0   
3       0            1    0    1       0        0    1        1        0   

   péssimo  qualidade  quebrado  recomendo  rápida  
0        0          1         0          0       1  
1        0          1         1          0       0  
2        0          0         0          1       1  
3        1          0         0          0       0  
Shape: (4, 14) | Não-zeros: 21
Esparsidade: 62.50%


In [29]:
#Célula 18: Matriz esparsa no modelo (Bloco 05, ✏️ corrigido)
#O y agora existe. A regressão logística aceita a matriz esparsa diretamente.

print(type(X))
y = [1, 0, 1, 0]   # 1 = positivo, 0 = negativo
model = LogisticRegression().fit(X, y)
print('Modelo treinado ✅')

<class 'scipy.sparse._csr.csr_matrix'>
Modelo treinado ✅


In [30]:
#Célula 19: N-grams (Blocos 06, 07 e 08)
#Trigramas capturam expressões como "de jeito nenhum". A seguir, compara quantas features cada ngram_range gera.

vec = CountVectorizer(ngram_range=(3, 3))
vec.fit(['não gostei nada do produto', 'muito bom mesmo, recomendo', 'foi de jeito nenhum'])
print(vec.get_feature_names_out())

vec = CountVectorizer(ngram_range=(1, 2), min_df=2)
vec.fit(corpus)
print('Uni+bi (min_df=2):', vec.get_feature_names_out())

corpus = ['Adorei o produto, super recomendo', 'Não gostei muito do produto',
          'Produto bom mesmo, vale o preço', 'Muito ruim, não comprem']
for n in [(1, 1), (1, 2), (1, 3), (2, 2)]:
    print(f'ngram_range={n}: {CountVectorizer(ngram_range=n).fit_transform(corpus).shape[1]} features')


['bom mesmo recomendo' 'de jeito nenhum' 'foi de jeito' 'gostei nada do'
 'muito bom mesmo' 'nada do produto' 'não gostei nada']
Uni+bi (min_df=2): ['atendimento' 'bom' 'entrega' 'entrega rápida' 'produto' 'qualidade'
 'rápida']
ngram_range=(1, 1): 14 features
ngram_range=(1, 2): 28 features
ngram_range=(1, 3): 38 features
ngram_range=(2, 2): 14 features


In [31]:
#Célula 20: Corpus real do Olist (🆕 nova)
#Antes de rodar, faça o upload de olist_order_reviews_dataset.csv na pasta /content/. Esta célula cria o reviews_corpus e o labels usados nas Células 21, 22, 25 e 26.

rev = pd.read_csv('/content/gdrive/MyDrive/Trabalhos em grupo/TECH CHALLENGE 4 - Entrega - 17 11/DATASETS/olist_order_reviews_dataset.csv')
rev = rev.dropna(subset=['review_comment_message'])
rev = rev[rev.review_score != 3]   # tira as notas neutras

reviews_corpus = rev.review_comment_message.apply(limpar_normalizar).tolist()
labels = (rev.review_score >= 4).astype(int).tolist()

print(f'{len(reviews_corpus)} reviews | positivas: {sum(labels)/len(labels):.0%}')


37420 reviews | positivas: 71%


In [32]:
#Célula 21: Filtros min_df e max_df (Bloco 09, ✏️ corrigido)
#Remove os termos raros e os onipresentes. Agora roda com dados reais.

vec = CountVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.85, max_features=10000)
X = vec.fit_transform(reviews_corpus)
print(f'Features: {X.shape[1]}')


Features: 10000


In [33]:
#Célula 22: Unigramas x bigramas (Bloco 10)
#Mede quanto os bigramas melhoram o F1 na análise de sentimento. ⏳ Leva cerca de 1 minuto.

X_uni = CountVectorizer(ngram_range=(1, 1), min_df=5).fit_transform(reviews_corpus)
X_bi  = CountVectorizer(ngram_range=(1, 2), min_df=5).fit_transform(reviews_corpus)
f1_uni = cross_val_score(LogisticRegression(max_iter=1000), X_uni, labels, cv=5, scoring='f1').mean()
f1_bi  = cross_val_score(LogisticRegression(max_iter=1000), X_bi,  labels, cv=5, scoring='f1').mean()
print(f'Unigramas: {f1_uni:.3f} | Uni+bi: {f1_bi:.3f} | Ganho: {(f1_bi - f1_uni)*100:+.1f} pts')


Unigramas: 0.947 | Uni+bi: 0.949 | Ganho: +0.2 pts


In [34]:
#Célula 23: TF e IDF manuais (Blocos 11 e 12)
#TF: frequência da palavra dentro do documento.
#IDF: quão rara a palavra é no corpus. Uma palavra presente em todos os documentos recebe IDF zero.

def tf(p, doc):
    w = doc.split()
    return w.count(p) / len(w)

def idf(p, corpus):
    df = sum(1 for d in corpus if p in d.split())
    return np.log(len(corpus) / df) if df > 0 else 0

print(tf('produto', 'o produto bom o produto rápido'))
c = ['o produto bom', 'o produto ruim', 'o produto quebrado', 'o produto rápido']
print(idf('o', c), idf('bom', c))


0.3333333333333333
0.0 1.3862943611198906


In [35]:
#Célula 24: Matriz TF-IDF manual (Bloco 13)
#Multiplica TF por IDF. "quebrado" aparece como o termo mais discriminativo do Doc 2.

corpus = ['produto bom rapida entrega', 'produto ruim quebrado',
          'entrega rapida atendimento bom', 'atendimento ruim produto bom']
vocab = sorted(set(p for d in corpus for p in d.split()))
matriz = [[tf(p, d) * idf(p, corpus) for p in vocab] for d in corpus]
print(pd.DataFrame(matriz, columns=vocab).round(3))



   atendimento    bom  entrega  produto  quebrado  rapida   ruim
0        0.000  0.072    0.173    0.072     0.000   0.173  0.000
1        0.000  0.000    0.000    0.096     0.462   0.000  0.231
2        0.173  0.072    0.173    0.000     0.000   0.173  0.000
3        0.173  0.072    0.000    0.072     0.000   0.000  0.173


In [36]:
#Célula 25: TfidfVectorizer em produção (Bloco 15)
#Gera o TF-IDF do corpus real e mostra os 5 termos de maior peso no primeiro review.

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.85,
                             max_features=10000, sublinear_tf=True)
X = vectorizer.fit_transform(reviews_corpus)
print(X.shape, '| não-zeros:', X.nnz)

names = vectorizer.get_feature_names_out()
doc0 = X[0].toarray().flatten()
print('Review 0:', reviews_corpus[0])
for i in np.argsort(doc0)[-5:][::-1]:
    print(f'  {names[i]}: {doc0[i]:.3f}')


(37420, 10000) | não-zeros: 539296
Review 0: recebi bem antes do prazo estipulado
  recebi bem: 0.516
  prazo estipulado: 0.414
  estipulado: 0.408
  bem antes: 0.322
  bem: 0.249


In [37]:
#Célula 26: BoW x TF-IDF, o duelo final (Bloco 16) 🏆
#Compara as duas representações com o mesmo modelo. Em geral o TF-IDF ganha.

X_bow   = CountVectorizer(ngram_range=(1, 2), min_df=5).fit_transform(reviews_corpus)
X_tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True).fit_transform(reviews_corpus)

for nome, Xv in [('BoW', X_bow), ('TF-IDF', X_tfidf)]:
    f1 = cross_val_score(LogisticRegression(max_iter=1000), Xv, labels, cv=5, scoring='f1').mean()
    print(f'{nome:7s} F1: {f1:.3f}')


BoW     F1: 0.949
TF-IDF  F1: 0.953
